<a href="https://colab.research.google.com/github/RautRitesh/slm_project/blob/main/Testing_finetuned_slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch accelerate

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Define your model repository
model_id = "zeri000/nepali_legal_qwen_merged_3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading 1.5B model directly into Colab CPU RAM...")
print("(This may take 2-3 minutes. The session RAM will increase by ~6GB)")

# 2. Load the model strictly on CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",               # Forces the model onto the CPU
    torch_dtype=torch.float32,      # Standard 32-bit float is most stable for CPU math
    low_cpu_mem_usage=True          # Optimizes RAM allocation during the download/load phase
)
print(" Model successfully loaded into CPU memory!\n")

# 3. Define the Generation Function
def ask_nepali_legal_ai(question: str) -> str:
    print(f"Generating response for: '{question}'...")
    start_time = time.time()

    # Apply your exact training ChatML format
    messages = [
        {"role": "system", "content": "तपाईं एक विशेषज्ञ नेपाली कानूनी सहायक हुनुहुन्छ।"},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize input (No .to("cuda") used here)
    inputs = tokenizer([prompt], return_tensors="pt")

    # Generate the output
    # Note: We keep max_new_tokens relatively low (150) so you don't wait forever on CPU
    with torch.no_grad(): # Disables gradient calculation to save memory/speed up inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            use_cache=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only the newly generated text
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    end_time = time.time()
    print(f"⏱️ Generation took {round(end_time - start_time, 2)} seconds.\n")

    return response
question_1 = "मुलुकी अपराध संहिता के हो?"
answer_1 = ask_nepali_legal_ai(question_1)
print(f"🤖 AI: {answer_1}\n")
print("-" * 50)

question_2 = "सम्बन्ध विच्छेद गर्न के के चाहिन्छ?"
answer_2 = ask_nepali_legal_ai(question_2)
print(f"🤖 AI: {answer_2}\n")

In [ ]:
answer=ask_nepali_legal_ai("सम्बन्ध विच्छेद भनेको के हो?")
print(answer)

Generating response for: 'सम्बन्ध विच्छेद भनेको के हो?'...
⏱️ Generation took 124.58 seconds.

यहाँको प्रश्नको सन्दर्भमा, सम्बन्ध विच्छेदले मध्यवर्ती पति/अभिमानी र मध्यवर्ती पत्नीबीच आफ्नो जीवन अवस्था घटेको उद्देश्यले गरिएको विवाहलाई जनाउँछ।


In [ ]:
def ask_nepali_legal_ai_2(question: str) -> str:
    print(f"Generating response for: '{question}'...")
    start_time = time.time()

    messages = [
        {"role": "system", "content": "तपाईं एक विशेषज्ञ नेपाली कानूनी सहायक हुनुहुन्छ।"},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt")

    # NEW: Define the specific stopping tokens for Qwen/ChatML
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>")
    ]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,           # ⬆️ INCREASED: Gives it enough room for a full answer
            use_cache=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=terminators      # 🛑 NEW: Tells the model exactly when to stop naturally
        )

    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    end_time = time.time()
    print(f"⏱️ Generation took {round(end_time - start_time, 2)} seconds.\n")

    return response

In [ ]:
answer=ask_nepali_legal_ai_2("नेपालमा सम्बन्ध विच्छेद गर्ने प्रक्रिया कस्तो हुन्छ?")
print(answer)

Generating response for: 'नेपालमा सम्बन्ध विच्छेद गर्ने प्रक्रिया कस्तो हुन्छ?'...
⏱️ Generation took 274.32 seconds.

यस विषयमा कानुनले स्पष्ट व्यवस्था गरेको छ: नेपालमा सम्बन्ध विच्छेद गर्ने प्रक्रियामा मध्यस्थ (मदरखण्ड) र सबैभन्दा फर्कने जस्तै आफूले उल्लेख गरेको छ, अघि भएको सम्झौतासँग सम्बन्धित ठानेको व्यक्तिले त्यस्तो सम्झौताको इच्छुक घोषणा गरी त्यस्तो सम्झौताको विघटन गर्न सक्नेछ। यदि त्यस्तो सम्झौता त्यस्तो व्यक्तिले त्यस्तो इच्छा दिएको छ भने, तिनीहरूले त्यस्तो सम्झौताको विघटन गर्नुहोस्।
